[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/history/h11_pi_curious.ipynb)

# The curious case of pi

Four thousand years of chasing one number, to try with your own hands: line up the old guesses against pi, peel
pi into its counts and find why 355/113 is so good, drop needles on a floor to find pi with no circle, check
Lazzarini's suspicious result against honest runs, and make William Shanks's mistake on purpose.

**How to use this notebook.** The grey boxes are cells. Click the first one, then press **Shift + Enter** (or the
▶ Run button) to run it. Run each cell in turn, from the top to the bottom. When a slider or a box appears, change
it and watch. Nothing can break: if something looks odd, run everything again from the top. It works in your
browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, and a few helpers. Run this cell first.
import math
import random
from fractions import Fraction
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Text, ToggleButton

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "figure.dpi": 100,
})

def arctan_inv(x, unity):
    """arctan(1/x) × unity, in whole numbers, by its endless sum (as Machin and Shanks worked it)."""
    total = term = unity // x
    x2, n, sign = x * x, 1, -1
    while term:
        term //= x2
        n += 2
        total += sign * (term // n)
        sign = -sign
    return total

def pi_digits(n):
    """π to n decimals as a string "31415…", from Machin's formula: π/4 = 4 arctan(1/5) − arctan(1/239)."""
    unity = 10 ** (n + 10)
    return str(4 * (4 * arctan_inv(5, unity) - arctan_inv(239, unity)) // 10 ** 10)

PI = Fraction(int(pi_digits(40)), 10 ** 40)    # exact enough for everything below
assert pi_digits(15) == "3141592653589793"
print("Ready. π =", pi_digits(30)[0] + "." + pi_digits(30)[1:], "…  Run the next cell.")

## 1 · The old guesses, side by side

Each bar is how far a guess sits from pi. Bars above the line are too big, bars below are too small. The clay
tablet from Susa said 3 1/8; the Rhind papyrus implies 256/81; Archimedes trapped pi between 223/71 and 22/7; Zu
Chongzhi gave 355/113.

In [ ]:
GUESSES = [("3", Fraction(3)), ("Susa tablet 25/8", Fraction(25, 8)), ("Rhind 256/81", Fraction(256, 81)),
           ("Archimedes 223/71", Fraction(223, 71)), ("Archimedes 22/7", Fraction(22, 7)), ("Zu 355/113", Fraction(355, 113))]
fig, ax = plt.subplots(figsize=(11, 4))
for i, (name, g) in enumerate(GUESSES):
    err = float(g - PI)
    ax.bar(i, err, color=CORAL if "355" in name else TEAL)
    ax.text(i, err + (0.004 if err >= 0 else -0.012), f"{err:+.7f}", ha="center", fontsize=10, color=CHALK)
ax.axhline(0, color=SAGE, lw=2)
ax.set_xticks(range(len(GUESSES)), [n for n, _ in GUESSES], rotation=15)
ax.set_ylabel("guess − π"); ax.set_title("how far each old guess is from π")
plt.show()
for name, g in GUESSES:
    print(f"{name:20s} {float(g):.10f}   off by {abs(float(g - PI)):.10f}")

## 2 · Peel a number into its counts

Take away the whole part. See how many times the leftover fits into one. Keep that count, and do the same with the
new leftover. For pi the counts begin 7, 15, 1, 292. Stopping just before a big count gives a very good fraction.
Type any number (like 3.14159265358979, 2.718281828459045 or 1.41421356237) and watch its counts and fractions.

In [ ]:
def peel(number="3.14159265358979323846"):
    try:
        x = Fraction(number.strip())
    except ValueError:
        print("Type a decimal number, like 3.14159265358979."); return
    counts, y = [], x
    for _ in range(8):
        q = y.numerator // y.denominator
        counts.append(q)
        if y == q: break
        y = 1 / (y - q)
    print("counts:", counts[0], ";", ", ".join(map(str, counts[1:])))
    print("fractions from stopping after each count:")
    for k in range(1, len(counts) + 1):
        f = Fraction(counts[k - 1])
        for c in reversed(counts[:k - 1]):
            f = c + 1 / f
        print(f"  {str(f):>16s} = {float(f):.12f}   off by {abs(float(f - x)):.2e}")

interact(peel, number=Text(value="3.14159265358979323846", description="number", continuous_update=False));

## 3 · The best fraction for every bottom number

For each bottom number up to the slider, take the best top number and see how close the fraction gets to pi (the
height is the number of correct decimals). The coral spike is 355/113: nothing beats it until 16,604.

In [ ]:
def best(max_bottom=500):
    xs, ys = [], []
    for d in range(1, max_bottom + 1):
        n = round(PI * d)
        err = abs(Fraction(n, d) - PI)
        xs.append(d); ys.append(-math.log10(float(err)))
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(xs, ys, color=TEAL, lw=1)
    if max_bottom >= 113:
        ax.scatter([113], [ys[112]], color=CORAL, s=60, zorder=3)
        ax.annotate("355/113", (113, ys[112]), xytext=(10, -6), textcoords="offset points", color=CORAL)
    ax.set_xlabel("bottom number"); ax.set_ylabel("correct decimals (about)")
    ax.set_title("how close the best fraction gets, for each bottom number")
    plt.show()

interact(best, max_bottom=IntSlider(value=500, min=10, max=3000, step=10, description="up to"));

## 4 · Pi with no circle: Buffon's needle

A floor of boards, each as wide as a needle is long. Drop needles at random. About 2/π of them land across a crack,
so 2 × drops ÷ crossings is close to pi. Change the number of drops and the seed (a different random run), and see
how close you get. Big runs land closer, but slowly.

In [ ]:
def buffon(drops=10000, seed=2026):
    rng = random.Random(seed)
    cross, est = 0, []
    for k in range(1, drops + 1):
        y, angle = rng.random() * 0.5, rng.random() * math.pi / 2
        cross += y <= 0.5 * math.sin(angle)
        if k % max(1, drops // 200) == 0: est.append((k, 2 * k / max(1, cross)))
    print(f"{drops:,} drops, {cross:,} cross a crack:  π ≈ 2 × {drops:,} ÷ {cross:,} = {2 * drops / max(1, cross):.4f}")
    fig, ax = plt.subplots(figsize=(11, 3.6))
    ax.plot([k for k, _ in est], [e for _, e in est], color=TEAL)
    ax.axhline(math.pi, color=CORAL, lw=2, label="π")
    ax.set_ylim(2.8, 3.5); ax.set_xlabel("drops so far"); ax.set_ylabel("estimate"); ax.legend()
    ax.set_title("the running estimate")
    plt.show()

interact(buffon, drops=IntSlider(value=10000, min=100, max=100000, step=100, description="drops"),
         seed=IntSlider(value=2026, min=1, max=5000, description="seed"));

## 5 · Lazzarini's suspicious answer

In 1901 Mario Lazzarini said he dropped 3,408 needles (each 5/6 of the gap) and saw 1,808 crossings, which gives
exactly 355/113. Here are 300 honest runs of the same size. How often does an honest run land as close to pi as
he did? Press the button to run them.

In [ ]:
def lazzarini(run_them=False):
    est = 2 * Fraction(5, 6) * 3408 / 1808
    print(f"Lazzarini: 2 × 5/6 × 3,408 ÷ 1,808 = {est} = {float(est):.7f}  (off by {abs(float(est - PI)):.1e})")
    print("3,408 = 16 × 213 and 1,808 = 16 × 113")
    if not run_them:
        print("Press the button to run 300 honest runs of 3,408 drops."); return
    rng, results = random.Random(1901), []
    for _ in range(300):
        cross = 0
        for _ in range(3408):
            y, angle = rng.random() * 0.5, rng.random() * math.pi / 2
            cross += y <= (5 / 12) * math.sin(angle)
        results.append(2 * (5 / 6) * 3408 / max(1, cross))
    close = sum(abs(r - math.pi) < 1e-6 for r in results)
    print(f"honest runs within a millionth of π: {close} of 300")
    fig, ax = plt.subplots(figsize=(11, 3.6))
    ax.hist(results, bins=30, color=TEAL)
    ax.axvline(float(est), color=CORAL, lw=3, label="Lazzarini")
    ax.set_xlabel("estimate of π"); ax.set_title("300 honest runs of 3,408 drops"); ax.legend()
    plt.show()

interact(lazzarini, run_them=ToggleButton(value=False, description="run 300 honest runs"));

## 6 · Make Shanks's mistake

William Shanks worked Machin's two sums by hand and left out terms. Leave out one term of the first sum here (pick
which), and see where the digits stop matching the true ones.

In [ ]:
def slip(term_left_out=60):
    n = 707
    unity = 10 ** (n + 10)
    total = term = unity // 5
    k, sign, x2, parts = 0, -1, 25, []
    while term:                                   # arctan(1/5), term by term, skipping one
        if k != term_left_out: parts.append(term // (2 * k + 1) * (1 if k % 2 == 0 else -1))
        term //= x2; k += 1
    wrong = str(4 * (4 * sum(parts) - arctan_inv(239, unity)) // 10 ** 10)
    right = pi_digits(n)
    first = next((i for i, (a, b) in enumerate(zip(right, wrong)) if a != b), None)
    if first is None:
        print("That term is too small to change the first 707 decimals: try a smaller number."); return
    print(f"leaving out term {term_left_out}: the digits agree up to decimal {first - 1}, then part at decimal {first}")
    print("true :", right[1:][max(0, first - 12):first + 8])
    print("slip :", wrong[1:][max(0, first - 12):first + 8])

interact(slip, term_left_out=IntSlider(value=60, min=1, max=100, description="leave out"));

## 7 · The challenge from the video

Write the first three odd numbers, each one twice: 1 1 3 3 5 5. Split the row in the middle. Which fraction
appears, and how close does it get to pi? Try it first, then press the button.

In [ ]:
def challenge(show_answer=False):
    if not show_answer:
        print("1 1 3 | 3 5 5  →  ?   (press the button when you're ready)"); return
    f = Fraction(355, 113)
    print(f"355 ÷ 113 = {float(f):.10f}…   π = {float(PI):.10f}…")
    print(f"six decimals right, out by {abs(float(f - PI)):.2e}: Zu Chongzhi's fraction, about 1,500 years old")

interact(challenge, show_answer=ToggleButton(value=False, description="show the answer"));

## Sources
Susa tablet (Old Babylonian); Rhind papyrus problem 50 (British Museum EA 10057); Archimedes, *Measurement of a
Circle*; Zu Chongzhi (c. 480); John Machin (1706) in William Jones's *Synopsis Palmariorum Matheseos*; William
Shanks (1873) and D. F. Ferguson (*Mathematical Gazette*, 1946); Buffon (1733, published 1777); M. Lazzarini (1901)
and L. Badger, *Mathematics Magazine* 67 (1994). All retrieved 2026-11-15. Every number above is computed in the
cells.